# 2D pose estimation — heatmap regression

**Track A · Human Modeling** · maps to lesson A2 (2D→3D pose).

The standard way to find joints: a CNN predicts one **heatmap per joint** (a blob at the joint), and **soft-argmax** turns each heatmap into a coordinate. We train it on a synthetic articulated arm so it is fully self-contained.

> CPU is fine.

In [ ]:
import os, torch, torch.nn as nn, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 1500)); H = W = 48; K = 3
ys, xs = torch.meshgrid(torch.arange(H, dtype=torch.float32), torch.arange(W, dtype=torch.float32), indexing="ij")
ys, xs = ys.to(device), xs.to(device)

## 1 · Synthetic data — a 3-joint arm (base · elbow · hand)

In [ ]:
def make_batch(n):                                       # vectorised: a 3-joint arm
    bx = W * 0.5 + torch.randn(n, 1, device=device) * 3
    by = H * 0.85 + torch.randn(n, 1, device=device) * 1.5
    a1 = -1.57 + (torch.rand(n, 1, device=device) - 0.5) * 1.4
    ex = bx + 14 * torch.cos(a1); ey = by + 14 * torch.sin(a1)
    a2 = a1 + (torch.rand(n, 1, device=device) - 0.5) * 1.6
    hx = ex + 12 * torch.cos(a2); hy = ey + 12 * torch.sin(a2)
    cx = torch.cat([bx, ex, hx], 1); cy = torch.cat([by, ey, hy], 1)        # (n,K)
    co = torch.stack([cx, cy], -1)                                          # (n,K,2)
    d2 = (xs[None, None] - cx[..., None, None]) ** 2 + (ys[None, None] - cy[..., None, None]) ** 2
    hms = torch.exp(-d2 / (2 * 2.0 ** 2))                                   # (n,K,H,W) target heatmaps
    imgs = hms.sum(1, keepdim=True).clamp(0, 1) + 0.03 * torch.randn(n, 1, H, W, device=device)
    return imgs, hms, co
xb, hb, cb = make_batch(4)
plt.imshow(xb[0, 0].cpu()); plt.title("input (joint blobs + noise)"); plt.axis("off"); plt.show()

## 2 · Model — a small fully-convolutional net (keeps resolution)

In [ ]:
class PoseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, K, 3, padding=1))
    def forward(self, x): return self.net(x)
def soft_argmax(hm):                          # differentiable coordinate decoder
    B, Kk, h, w = hm.shape
    p = torch.softmax(hm.reshape(B, Kk, -1), -1).reshape(B, Kk, h, w)
    return torch.stack([(p * xs).sum((-1, -2)), (p * ys).sum((-1, -2))], -1)
def coords_from(hm):                          # hard peak location -> (B,K,2) as (x,y), for PCK
    B, Kk, h, w = hm.shape
    idx = hm.reshape(B, Kk, -1).argmax(-1)
    return torch.stack([idx % w, idx // w], -1).float()

## 3 · Train — MSE on the heatmaps

In [ ]:
model = PoseNet().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); hist = []
for step in range(STEPS + 1):
    x, hm, co = make_batch(16)
    pred = model(x); loss = ((pred - hm) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0:
        with torch.no_grad():
            pck = ((coords_from(pred) - co).norm(dim=-1) < 0.1 * (H ** 2 + W ** 2) ** 0.5).float().mean().item()
        hist.append((step, round(pck, 3))); print(f"step {step:4d}  loss {loss.item():.4f}  PCK@0.1 {pck:.3f}")

## 4 · Compare — predicted joints vs. ground truth

In [ ]:
x, hm, co = make_batch(1)
with torch.no_grad(): pr = model(x); pj = coords_from(pr)[0].cpu()
fig, ax = plt.subplots(1, 2, figsize=(7, 3.6))
ax[0].imshow(x[0, 0].cpu()); ax[0].scatter(co[0, :, 0].cpu(), co[0, :, 1].cpu(), c="lime", s=40, label="truth")
ax[0].scatter(pj[:, 0], pj[:, 1], c="red", marker="x", s=40, label="pred"); ax[0].legend(); ax[0].set_title("joints"); ax[0].axis("off")
ax[1].imshow(pr[0].sum(0).cpu()); ax[1].set_title("predicted heatmaps"); ax[1].axis("off")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/A_pose_heatmap/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/A_pose_heatmap"; os.makedirs(run, exist_ok=True)
torch.save(model.state_dict(), f"{run}/pose.pt")
json.dump({"pck": hist}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/ropedia-" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
2D joints from first-person views feed **C · Egocentric** understanding.

### Where to go next
- Add a U-Net / stacked-hourglass and real images (MPII, COCO).
- **Lift** these 2D joints to 3D with a small lifter network (lesson A2), or fit SMPL to them (the SMPLify lab).